# Phase 2 — Data Cleaning

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 50)

PROC_DIR = Path('../data/processed')
df = pd.read_csv(PROC_DIR / 'olist_merged_raw.csv')
print('Loaded shape:', df.shape)
df.head()


Loaded shape: (112650, 38)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,n_payment_methods,review_score,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02-10-2017 10:56,02-10-2017 11:07,04-10-2017 19:55,10-10-2017 21:25,18-10-2017 00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350,maua,SP,38.71,1.0,voucher,2.0,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24-07-2018 20:41,26-07-2018 03:24,26-07-2018 14:31,07-08-2018 15:27,13-08-2018 00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570,belo horizonte,SP,141.46,1.0,boleto,1.0,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08-08-2018 08:38,08-08-2018 08:55,08-08-2018 13:50,17-08-2018 18:06,04-09-2018 00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840,guariba,SP,179.12,3.0,credit_card,1.0,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,18-11-2017 19:28,18-11-2017 19:45,22-11-2017 13:39,02-12-2017 00:28,15-12-2017 00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842,belo horizonte,MG,72.20,1.0,credit_card,1.0,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,13-02-2018 21:18,13-02-2018 22:20,14-02-2018 19:46,16-02-2018 18:17,26-02-2018 00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752,mogi das cruzes,SP,28.62,1.0,credit_card,1.0,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


### Missing value handling

In [2]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_report[missing_report['missing'] > 0]


,missing,pct
review_comment_message,65251,57.92
order_delivered_customer_date,2454,2.18
product_category_name_english,1627,1.44
product_name_lenght,1603,1.42
product_photos_qty,1603,1.42
product_category_name,1603,1.42
product_description_lenght,1603,1.42
order_delivered_carrier_date,1194,1.06
review_score,942,0.84
review_answer_timestamp,942,0.84


### Duplicate removal

In [3]:
before = len(df)
df = df.drop_duplicates()
print(f'Dropped {before - len(df)} exact duplicate rows -> {len(df):,} rows remain')


Dropped 0 exact duplicate rows -> 112,650 rows remain


### Date conversion

In [4]:
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'review_creation_date',
    'review_answer_timestamp',
]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

df[date_cols].dtypes


C:\Users\Pavithran\AppData\Local\Temp\ipykernel_1128\3502524113.py:8: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[c] = pd.to_datetime(df[c], errors='coerce')


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_creation_date             datetime64[us]
review_answer_timestamp          datetime64[us]
dtype: object

### Invalid record handling

In [5]:
# Orders must have a purchase timestamp to be usable
before = len(df)
df = df[df['order_purchase_timestamp'].notna()]
print(f'Dropped {before - len(df)} rows with no purchase timestamp')

# Delivered date can't be before purchase date
bad_dates = df['order_delivered_customer_date'] < df['order_purchase_timestamp']
print(f'Dropping {bad_dates.sum()} rows with delivery date before purchase date')
df = df[~bad_dates]

# Negative or zero price/freight are invalid for a real transaction
bad_price = (df['price'] <= 0) | (df['freight_value'] < 0)
print(f'Dropping {bad_price.sum()} rows with non-positive price or negative freight')
df = df[~bad_price]

print('Shape after invalid-record removal:', df.shape)


Dropped 68078 rows with no purchase timestamp
Dropping 1848 rows with delivery date before purchase date
Dropping 0 rows with non-positive price or negative freight
Shape after invalid-record removal: (42724, 38)


### Outlier detection

In [6]:
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ['price', 'freight_value', 'payment_value']:
    lo, hi = iqr_bounds(df[col])
    n_outliers = ((df[col] < lo) | (df[col] > hi)).sum()
    df[f'{col}_is_outlier'] = ((df[col] < lo) | (df[col] > hi)).astype(int)
    print(f'{col:16s} bounds=({lo:.2f}, {hi:.2f})  outliers={n_outliers} ({n_outliers/len(df)*100:.2f}%)')

# We FLAG outliers rather than drop them — high-value purchases are business-
# relevant signal (they matter a lot for CLV), not necessarily data errors.


price            bounds=(-251.04, 428.03)  outliers=1598 (3.74%)
freight_value    bounds=(-11.25, 45.45)  outliers=2090 (4.89%)
payment_value    bounds=(-326.65, 589.22)  outliers=1938 (4.54%)


In [7]:
# df.dtypes

### Feature consistency checks

In [8]:
assert df['order_id'].notna().all(), "order_id must never be null"
assert df['customer_unique_id'].notna().all(), "customer_unique_id must never be null"
assert (df['price'] > 0).all(), "price must be positive"
assert (df['freight_value'] >= 0).all(), "freight_value must be non-negative"
assert df['order_purchase_timestamp'].notna().all(), "purchase timestamp required"

print('All consistency checks passed.')
print('Final shape:', df.shape)
print('Unique customers (customer_unique_id):', df['customer_unique_id'].nunique())
print('Unique orders:', df['order_id'].nunique())


All consistency checks passed.
Final shape: (42724, 41)
Unique customers (customer_unique_id): 36950
Unique orders: 37618


### Save cleaned dataset

In [9]:
out_path = PROC_DIR / 'olist_merged_clean.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}  ({df.shape[0]:,} rows, {df.shape[1]} cols)')


Saved to ..\data\processed\olist_merged_clean.csv  (42,724 rows, 41 cols)
